# Agent offline — pokaz szkoleniowy
Python 3.10+, bez API i dodatkowych pakietów. Polityka jest regułowa, bez LLM.
Uruchamiaj komórki od góry. Notebook jest samodzielny. Tworzy własny folder tymczasowy.


In [ ]:
from pathlib import Path
import tempfile
import json

# Nowy folder przy każdym uruchomieniu; nie zastępujemy Twoich plików.
DEMO_ROOT = Path(tempfile.mkdtemp(prefix="agent_szkolenie_"))
(DEMO_ROOT / "dane").mkdir()
(DEMO_ROOT / "dane/sprzedaz.csv").write_text('produkt,ilosc,cena,koszt\nSzkolenie Python,10,500,200\nAudyt AI,3,2000,800\nWarsztat ML,8,700,300\nSzkolenie Python,5,500,200\n', encoding="utf-8")
(DEMO_ROOT / "dane/wiedza.md").write_text('Agent to system, który odbiera cel lub obserwacje, wybiera działania i korzysta z ich wyników, aby wykonać zadanie. W tym przykładzie polityka wyboru działań jest regułowa. Agent nie musi używać LLM.\n\nLLM to duży model językowy uczony na danych tekstowych. Może proponować działania agenta, ale potrzebuje osobnej warstwy wykonującej i kontrolującej narzędzia. Ten program nie zawiera LLM.\n\nRAG łączy wyszukiwanie informacji z generowaniem odpowiedzi przez model. Nasze wyszukiwanie zwraca gotowe akapity ze źródłami; bez generatora LLM nie jest pełnym systemem RAG.\n\nPamięć agenta może przechowywać wyniki wcześniejszych działań. W tym programie pamięć zawiera ostatni wynik kalkulatora i znika po zakończeniu procesu. Log JSON można zachować osobno, lecz agent nie wczytuje go automatycznie.\n\nMarża kwotowa produktu w demonstracji to ilość sprzedanych sztuk pomnożona przez różnicę ceny jednostkowej i kosztu jednostkowego. To uproszczony wskaźnik demonstracyjny, bez kosztów stałych, podatków i zwrotów.\n', encoding="utf-8")
print("Folder pokazu:", DEMO_ROOT)


## Kod źródłowy
To ten sam kod co w `agent.py`; pominięto tylko automatyczny start interfejsu terminalowego.
Przejdź do `decide()` i `run()`, aby omówić pętlę agenta.


In [ ]:
"""Agent szkoleniowy: cel -> decyzja -> narzędzie -> obserwacja -> decyzja.

Python 3.10+, tylko biblioteka standardowa. Brak API i ruchu sieciowego.
Polityka jest regułowa, NIE jest to LLM ani model uczący się z rozmów.
"""
from __future__ import annotations

import argparse
import ast
import csv
import json
import math
import operator
import re
import unicodedata
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation
from pathlib import Path


def normalize(text: str) -> str:
    """Usuń polskie znaki wyłącznie na potrzeby dopasowania intencji."""
    text = text.lower().replace("ł", "l")
    return "".join(c for c in unicodedata.normalize("NFKD", text)
                   if not unicodedata.combining(c))


def calculate(expression: str) -> float:
    """Interpretuj wyłącznie liczby i wybrane operatory AST. Nigdy eval()."""
    if len(expression) > 200:
        raise ValueError("Wyrażenie jest za długie (maks. 200 znaków).")
    tree = ast.parse(expression.replace(",", "."), mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 70:
        raise ValueError("Wyrażenie jest zbyt złożone.")
    operations = {ast.Add: operator.add, ast.Sub: operator.sub,
                  ast.Mult: operator.mul, ast.Div: operator.truediv,
                  ast.Pow: operator.pow}

    def visit(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = float(node.value)
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = visit(node.operand) * (-1 if isinstance(node.op, ast.USub) else 1)
        elif isinstance(node, ast.BinOp) and type(node.op) in operations:
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 12:
                raise ValueError("Potęga musi mieć wykładnik od -12 do 12.")
            value = operations[type(node.op)](left, right)
        else:
            raise ValueError("Dozwolone: liczby, nawiasy, +, -, *, /, **.")
        if isinstance(value, complex) or not math.isfinite(value) or abs(value) > 1e15:
            raise ValueError("Wynik poza zakresem kalkulatora.")
        return value
    return visit(tree.body)


@dataclass
class State:
    """Pamięć jednego zadania: obserwacje oraz jawny dziennik działań."""
    goal: str
    observations: dict = field(default_factory=dict)
    trace: list = field(default_factory=list)


class Agent:
    def __init__(self, root: str | Path, max_steps: int = 6):
        if max_steps < 1:
            raise ValueError("max_steps musi być dodatnie.")
        self.root = Path(root).resolve()
        self.max_steps = max_steps
        self.memory = {}  # pamięć między zadaniami, tylko w tym procesie
        self.tools = {
            "kalkulator": self.calculator,
            "szukaj_wiedzy": self.search,
            "czytaj_sprzedaz": self.read_sales,
            "analizuj_sprzedaz": self.analyze_sales,
            "zapisz_raport": self.save_report,
        }

    def path(self, name: str) -> Path:
        path = (self.root / name).resolve()
        if not path.is_relative_to(self.root):
            raise ValueError("Plik musi znajdować się w folderze projektu.")
        return path

    def calculator(self, expression: str) -> dict:
        # Pozwala pokazać pamięć: po 'Oblicz: 100*2' -> 'Oblicz: wynik/4'.
        if re.search(r"\bwynik\b", expression, re.I):
            if "last_result" not in self.memory:
                raise ValueError("Nie mam poprzedniego wyniku obliczenia.")
            expression = re.sub(r"\bwynik\b", f"({self.memory['last_result']})",
                                expression, flags=re.I)
        result = calculate(expression)
        self.memory["last_result"] = result
        return {"expression": expression, "result": result}

    def search(self, query: str) -> dict:
        """Proste wyszukiwanie leksykalne; odpowiedź cytuje lokalne akapity."""
        stop = {"co", "to", "jest", "jak", "dziala", "czy", "i", "w", "o", "na"}
        tokens = set(re.findall(r"\w+", normalize(query))) - stop
        text = self.path("dane/wiedza.md").read_text(encoding="utf-8")
        hits = []
        for number, paragraph in enumerate(text.split("\n\n"), 1):
            words = set(re.findall(r"\w+", normalize(paragraph)))
            score = len(tokens & words) / max(len(tokens), 1)
            if score:
                hits.append({"score": score, "text": paragraph.strip(),
                             "source": f"dane/wiedza.md, akapit {number}"})
        hits.sort(key=lambda item: item["score"], reverse=True)
        return {"hits": hits[:2]}

    def read_sales(self, filename: str) -> dict:
        path = self.path(filename)
        if path.stat().st_size > 1_000_000:
            raise ValueError("Plik przekracza limit 1 MB.")
        with path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            required = {"produkt", "ilosc", "cena", "koszt"}
            if not required.issubset(set(reader.fieldnames or [])):
                raise ValueError("CSV musi mieć kolumny: produkt,ilosc,cena,koszt.")
            rows = []
            for line, row in enumerate(reader, 2):
                try:
                    name = row["produkt"].strip()
                    numbers = {key: Decimal(row[key]) for key in ("ilosc", "cena", "koszt")}
                    if not name or any(not n.is_finite() or n < 0 or n > 1_000_000_000
                                       for n in numbers.values()):
                        raise ValueError()
                    if numbers["ilosc"] != numbers["ilosc"].to_integral_value():
                        raise ValueError()
                except (ValueError, InvalidOperation, TypeError, AttributeError):
                    raise ValueError(f"Niepoprawne dane CSV w wierszu {line}.") from None
                rows.append({"produkt": name, **{k: str(v) for k, v in numbers.items()}})
        if not rows:
            raise ValueError("CSV nie zawiera danych.")
        return {"rows": rows, "source": filename}

    def analyze_sales(self, rows: list, metric: str) -> dict:
        """Wybierz produkt po przychodzie lub marży kwotowej, nie procentowej."""
        totals = {}
        for row in rows:
            item = totals.setdefault(row["produkt"], {"przychod": Decimal(0), "marza": Decimal(0)})
            quantity, price, cost = (Decimal(row[k]) for k in ("ilosc", "cena", "koszt"))
            item["przychod"] += quantity * price
            item["marza"] += quantity * (price - cost)
        highest = max(item[metric] for item in totals.values())
        leaders = sorted(name for name, item in totals.items() if item[metric] == highest)
        return {"metric": metric, "leaders": leaders,
                "totals": {name: {k: str(v) for k, v in item.items()}
                           for name, item in sorted(totals.items())}}

    @staticmethod
    def sales_answer(analysis: dict) -> str:
        metric = analysis["metric"]
        names = analysis["leaders"]
        value = Decimal(analysis["totals"][names[0]][metric])
        label = "marża kwotowa" if metric == "marza" else "przychód"
        return (f"Najwyższy wynik — {label}: {', '.join(names)}; {value:.2f} PLN. "
                "Marża kwotowa = ilość × (cena jednostkowa − koszt jednostkowy).")

    def save_report(self, analysis: dict) -> dict:
        path = self.path("wyniki/raport.md")
        path.parent.mkdir(parents=True, exist_ok=True)
        # Stała nazwa celowo: kolejne uruchomienie aktualizuje raport demo.
        lines = ["# Raport sprzedaży — dane demonstracyjne", "", self.sales_answer(analysis), ""]
        for name, values in analysis["totals"].items():
            lines.append(f"- {name}: przychód {values['przychod']} PLN; marża {values['marza']} PLN")
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        return {"path": str(path)}

    def decide(self, state: State) -> tuple[str, dict]:
        """Polityka agenta: następna akcja zależy od celu i OBSERWACJI.

        To miejsce można na zajęciach zastąpić polityką opartą na modelu.
        Zwracamy akcję i argumenty albo kończymy odpowiedzią.
        """
        goal = normalize(state.goal)
        seen = state.observations
        if "sprzedaz" in goal or "csv" in goal:
            if "czytaj_sprzedaz" not in seen:
                return "czytaj_sprzedaz", {"filename": "dane/sprzedaz.csv"}
            if "analizuj_sprzedaz" not in seen:
                return "analizuj_sprzedaz", {
                    "rows": seen["czytaj_sprzedaz"]["rows"],
                    "metric": "marza" if "marz" in goal else "przychod"}
            analysis = seen["analizuj_sprzedaz"]
            if "raport" in goal and "zapisz_raport" not in seen:
                return "zapisz_raport", {"analysis": analysis}
            answer = self.sales_answer(analysis)
            if "zapisz_raport" in seen:
                answer += " Raport: " + seen["zapisz_raport"]["path"]
            return "koniec", {"answer": answer}
        if goal.startswith(("oblicz:", "policz:")):
            if "kalkulator" not in seen:
                return "kalkulator", {"expression": state.goal.split(":", 1)[1].strip()}
            return "koniec", {"answer": f"Wynik: {seen['kalkulator']['result']:g}"}
        if goal.startswith(("wiedza:", "szukaj:")):
            if "szukaj_wiedzy" not in seen:
                return "szukaj_wiedzy", {"query": state.goal.split(":", 1)[1].strip()}
            hits = seen["szukaj_wiedzy"]["hits"]
            answer = "\n\n".join(f"{h['text']}\nŹródło: {h['source']}" for h in hits)
            return "koniec", {"answer": answer or "Nie znalazłem odpowiedzi w lokalnej bazie wiedzy."}
        return "koniec", {"answer": "Obsługuję: Oblicz: 2+2 | Wiedza: czym jest agent? | "
                          "Przeanalizuj sprzedaż według marży i zapisz raport."}

    def run(self, goal: str, verbose: bool = True) -> dict:
        state = State(goal=goal)
        answer = "Przerwano: osiągnięto limit kroków."
        status = "limit"
        for step in range(1, self.max_steps + 1):
            action, arguments = self.decide(state)
            if action == "koniec":
                answer, status = arguments["answer"], "ok"
                break
            if verbose:
                print(f"[{step}] Narzędzie: {action}")
            try:
                observation = self.tools[action](**arguments)
                state.observations[action] = observation
                state.trace.append({"step": step, "tool": action,
                                    "arguments": arguments, "observation": observation})
                if verbose:
                    preview = json.dumps(observation, ensure_ascii=False)
                    print("    Obserwacja:", preview[:220])
            except (OSError, ValueError, SyntaxError, ArithmeticError) as exc:
                answer, status = f"Nie ukończono zadania: {exc}", "error"
                state.trace.append({"step": step, "tool": action, "error": str(exc)})
                break
        result = {"goal": goal, "status": status, "answer": answer, "trace": state.trace}
        if verbose:
            print("Odpowiedź:", answer, "\n")
        return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("goal", nargs="?", help="Cel agenta w cudzysłowie")
    parser.add_argument("--demo", action="store_true", help="Uruchom pokaz czterech zadań")
    parser.add_argument("--chat", action="store_true", help="Tryb interaktywny")
    parser.add_argument("--log", help="Zapisz przebieg w JSON wewnątrz folderu projektu")
    args = parser.parse_args()
    agent = Agent(Path(__file__).resolve().parent)
    results = []
    if args.demo:
        goals = ["Oblicz: (1200 + 800) * 0.9", "Oblicz: wynik / 3",
                 "Wiedza: czym jest agent?",
                 "Przeanalizuj sprzedaż według marży i zapisz raport"]
        for goal in goals:
            print("CEL:", goal)
            results.append(agent.run(goal))
    elif args.chat:
        print("Agent offline. Wpisz 'koniec', aby zakończyć. Pomoc: wpisz 'pomoc'.")
        while True:
            try:
                goal = input("Ty> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if normalize(goal) in {"koniec", "exit", "quit"}:
                break
            results.append(agent.run(goal))
    elif args.goal:
        results.append(agent.run(args.goal))
    else:
        parser.print_help()
    if args.log:
        path = agent.path(args.log)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")



## 1. Kalkulator i pamięć sesji

In [ ]:
agent = Agent(DEMO_ROOT)
r1 = agent.run("Oblicz: (1200 + 800) * 0.9")
r2 = agent.run("Oblicz: wynik / 3")
assert r1["trace"][0]["observation"]["result"] == 1800
assert r2["trace"][0]["observation"]["result"] == 600


## 2. Wyszukiwanie lokalnych akapitów ze źródłem

In [ ]:
r3 = agent.run("Wiedza: czym jest RAG?")
r4 = agent.run("Wiedza: pingwiny")
assert "RAG łączy" in r3["answer"]
assert "Nie znalazłem" in r4["answer"]


## 3. Zadanie wieloetapowe: od danych do raportu

In [ ]:
r5 = agent.run("Przeanalizuj sprzedaż według marży i zapisz raport")
assert r5["status"] == "ok"
assert "4500.00" in r5["answer"]
print((DEMO_ROOT / "wyniki/raport.md").read_text(encoding="utf-8"))


## 4. Zmiana danych zmienia wybór produktu
Zwiększamy koszt Szkolenia Python w obu wierszach. Porównaj marżę i przychód.

In [ ]:
sales_path = DEMO_ROOT / "dane/sprzedaz.csv"
original = sales_path.read_text(encoding="utf-8")
try:
    sales_path.write_text(original.replace(",500,200", ",500,450"), encoding="utf-8")
    margin = agent.run("Przeanalizuj sprzedaż według marży")
    revenue = agent.run("Przeanalizuj sprzedaż według przychodu")
    assert margin["trace"][1]["observation"]["leaders"] == ["Audyt AI"]
    assert revenue["trace"][1]["observation"]["leaders"] == ["Szkolenie Python"]
finally:
    sales_path.write_text(original, encoding="utf-8")


## 5. Błędy i ograniczenie liczby działań

In [ ]:
error = agent.run("Oblicz: 1/0")
assert error["status"] == "error"
limited = Agent(DEMO_ROOT, max_steps=1).run("Przeanalizuj sprzedaż")
assert limited["status"] == "limit"
print("Wszystkie asercje pokazu zakończone sukcesem.")


## 6. Jawny zapis działań
Log zawiera narzędzia i wyniki, a nie ukryte rozumowanie modelu.

In [ ]:
print(json.dumps(r5["trace"], ensure_ascii=False, indent=2))
log_path = DEMO_ROOT / "wyniki/przebieg_notebook.json"
log_path.write_text(json.dumps([r1,r2,r3,r4,r5], ensure_ascii=False, indent=2), encoding="utf-8")
print("Zapisano:", log_path)


## Ćwiczenie
Dodaj narzędzie średniej oraz test pustej listy. Następnie rozważ wymianę `decide()` na politykę LLM.
Ten agent ma ograniczone intencje i nie interpretuje negacji; używaj pokazanych poleceń.
